# Nhóm 1 — MLP PyTorch (Kaggle)
Notebook adapter dùng chung `src/`/`scripts`, Seed=42, đọc input từ `/kaggle/input` và đóng gói artifact vào `/kaggle/working`.

In [ ]:
from pathlib import Path
import os, sys, shutil
PROJECT_ROOT = Path('/kaggle/working/BTL03')
INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
if not PROJECT_ROOT.exists():
    raise FileNotFoundError('Checkout repository vào /kaggle/working/BTL03 trước khi chạy notebook.')
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print('Kaggle:', WORK_ROOT.exists(), 'input:', INPUT_ROOT.exists())
from src.utils import seed_everything
seed_everything(42)

Đưa MNIST vào `data/raw/mnist/` nếu dataset đã được đính kèm. Nếu thiếu, script tải MNIST chính thức (torchvision nếu có; IDX fallback nếu không).

In [ ]:
!python scripts/download_data.py check

## T0201 — sweep Gaussian A/B/C và biểu đồ 3D

In [ ]:
!python scripts/run_group1_mlp.py --task func

In [ ]:
import pandas as pd
display(pd.read_csv('experiments/csv_logs/t0201_func_apxm_abc_sweep.csv'))
from IPython.display import Image, display
display(Image(filename='outputs/figures/t0201_gaussian_surface.png'))

## T0202 — huấn luyện MNIST, checkpoint và vector fc2

In [ ]:
!python scripts/run_group1_mlp.py --task mnist

In [ ]:
import json
metrics = json.load(open('outputs/features/mnist_mlp_metrics.json', encoding='utf-8'))
print({k:v for k,v in metrics.items() if k != 'history'})
%load_ext tensorboard
%tensorboard --logdir experiments/tensorboard/Group1_MLP

## Đóng gói artifact
Bao gồm mã/config, CSV, TensorBoard và outputs trong ZIP làm việc; dữ liệu và checkpoint vẫn bị Git ignore.

In [ ]:
import shutil
artifact = WORK_ROOT / 'BTL03_Group1_MLP_artifacts'
artifact.mkdir(exist_ok=True)
for item in ['experiments/csv_logs', 'experiments/tensorboard/Group1_MLP', 'outputs/checkpoints/01_mlp', 'outputs/features', 'outputs/figures']:
    source = PROJECT_ROOT / item
    if source.exists():
        target = artifact / item.replace('/', '_')
        if source.is_dir(): shutil.copytree(source, target, dirs_exist_ok=True)
        else: shutil.copy2(source, target)
zip_path = shutil.make_archive(str(WORK_ROOT / 'BTL03_Group1_MLP_artifacts'), 'zip', artifact)
print('Artifact:', zip_path)